# Joint midpoint transition survival models
Fit 52 TCAT GEPs plus reciprocal $\mathcal O_\lambda$ in one model, and 11 hard-label atlas cell types plus reciprocal $\mathcal O_\lambda$ in another. Each feature's crossover is fixed at its individual midpoint-model posterior median. The joint log hazard uses $\sum_j \beta_{dj}(q_{ij}-0.5)$ and integrates over patient particles. The HR for a feature compares its high and low plateaus conditional on the other features and covariates.

This notebook reads existing individual results. The path overrides below can point to their CSVs on Drive. An older uncentered fit or an incomplete set of crossovers will be reported before sampling. Select one representation to run it independently.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import numpy as np
import pandas as pd

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
RUN = ['gep', 'atlas']
# Optional exact paths to the midpoint regime_results.csv and O-lambda model_result.csv.
CROSSOVER_CSV = {'gep': None, 'atlas': None, 'olambda': None}
TCAT = DATA/'patient_TCAT_particles'
ATLAS = DATA/'Functional_CART_atlas_patient_particles_hard_labels_all8'
O = DATA/'olambda_reciprocal_direct_no_asymptote_floor_v1'
oe, op = O/'patient_olambda_asymptotes.csv', O/'patient_olambda_particles.csv'
outcomes = DATA/'clinical_outcomes.csv'
inputs = {
    'gep': (TCAT/'all_patient_TCAT_estimates_harmonized.csv', TCAT/'all_patient_TCAT_particles_harmonized.npz'),
    'atlas': (ATLAS/'patient_atlas_estimates.csv', ATLAS/'patient_atlas_particles.npz'),
}
source_roots = {'gep': DATA/'bayesian_TCAT_survival',
                'atlas': DATA/'bayesian_Functional_CART_atlas_survival',
                'olambda': DATA/'bayesian_Olambda_survival'/'direct_reciprocal_adjusted_midpoint_v1'/'smooth_transition_Q10_Q90_midpoint'}
OUTPUT = DATA/'bayesian_joint_direct_reciprocal_midpoint_v1'
draws, tune, chains, cores = 2000, 2000, 4, 4
width, target_accept, seed = 0.10, 0.99, 0
adjustments = {'Treatment':'axicel','Disease':'lymphoma'}
assert RUN and set(RUN) <= inputs.keys() and len(set(RUN)) == len(RUN)

In [ ]:
revision = 'fed8625240286b7e00352faf03898eaa5a072f17'
repo = Path(tempfile.mkdtemp(prefix='joint-midpoint-'))/'repo'
subprocess.run(['git','clone','-q','--single-branch','--branch','direct-olambda',
                'https://github.com/adamnik/CAR-T-spectral-organization.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','-q','--detach',revision],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pymc==5.28.5','arviz>=0.22,<1','h5netcdf>=1.5'],check=True)
print('Joint model revision:', revision)

In [ ]:
def digest(path):
    with open(path,'rb') as stream:
        h = hashlib.sha256()
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
        return h.hexdigest()

def confirmed_midpoint(m):
    offset = m.get('hazard_activation_offset')
    if offset is not None:
        return offset == 0.5
    return (m.get('baseline_reference') == 'transition midpoint' or
            'midpoint' in m.get('model','').lower())

def check_metadata(m):
    if not confirmed_midpoint(m) or m.get('adjustments') != adjustments:
        raise ValueError('Missing midpoint confirmation or incompatible clinical adjustments')
    prior = m.get('crossover_prior', {})
    low = prior.get('quantile_low', m.get('crossover_quantile_low'))
    high = prior.get('quantile_high', m.get('crossover_quantile_high'))
    if (low, high, m.get('transition_scale_sd')) != (0.10, 0.90, width):
        raise ValueError('Incompatible crossover prior or transition width')

def validate_source(path, name, expected):
    path = Path(path)
    table = pd.read_csv(path)
    root = path.parent
    manifest = root/'run_manifest.json'
    m = json.loads(manifest.read_text()) if manifest.exists() else {}
    if name == 'olambda':
        check_metadata(m)
        if len(table) != 1 or not 0 < float(table.iloc[0].crossover_median) < 1:
            raise ValueError('Expected one reciprocal crossover in (0,1)')
        config = json.loads((root/'notebook_config.json').read_text())
        if config['input_sha256'] != {p.name: digest(p) for p in (oe,op,outcomes)}:
            raise ValueError('O-lambda fit used different estimates, particles or outcomes')
    else:
        table = table.set_index('feature',verify_integrity=True)
        missing = sorted(set(expected)-set(table.index))
        if missing:
            raise ValueError(f'Missing {len(missing)} individual crossovers: {missing[:5]}')
        table = table.loc[expected]
        if not np.isfinite(table.crossover_median.to_numpy(float)).all():
            raise ValueError('Nonfinite crossover')
        if confirmed_midpoint(m):
            check_metadata(m)
        else:
            # Some individual-fit notebooks record the parameterization per feature.
            metas = {}
            for file in root.glob('individual/*/model_manifest.json'):
                fm = json.loads(file.read_text())
                metas[fm.get('feature')] = fm
            for feature in expected:
                check_metadata(metas.get(feature, {}))
    if not table.patients.eq(163).all() or not table.datasets.eq(8).all():
        raise ValueError('Expected completed individual fits on the full eight-dataset cohort')
    return path

def resolve_source(name, expected=None):
    if CROSSOVER_CSV[name] is not None:
        return validate_source(CROSSOVER_CSV[name], name, expected)
    root = source_roots[name]
    filename = 'model_result.csv' if name=='olambda' else 'regime_results.csv'
    valid, rejected = [], []
    for path in sorted(root.rglob(filename)):
        if any('running-' in part for part in path.parts):
            continue
        try:
            valid.append(validate_source(path,name,expected))
        except (KeyError, ValueError, OSError) as error:
            rejected.append(f'{path}: {error}')
    if len(valid) != 1:
        for msg in [*map(str,valid),*rejected]:
            print(msg)
        raise ValueError(f'{name}: {len(valid)} compatible tables under {root}. Set CROSSOVER_CSV[{name!r}] to a completed midpoint table.')
    return valid[0]

In [ ]:
keys = ['Dataset','sample_id']
est = pd.read_csv(oe).set_index(keys,verify_integrity=True)
assert len(est)==163 and est.olambda_scale.eq('reciprocal').all()
assert est.model.eq('bounded_direct_no_asymptote_floor').all()
assert ((est.O_lambda_inf_central>0)&(est.O_lambda_inf_central<1)).all()
p = pd.read_csv(op).set_index(keys,verify_integrity=True)
assert p.index.sort_values().equals(est.index.sort_values()) and p.shape[1]==16
assert ((p>0)&(p<1)).all().all()
o_cut = resolve_source('olambda')
print('O-lambda crossover:', o_cut, flush=True)
features, cuts = {}, {}
for name in RUN:
    csv, npz = inputs[name]
    with np.load(npz,allow_pickle=False) as archive:
        names = archive['feature_names'].astype(str).tolist()
    prefix = 'TCAT_GEP_' if name=='gep' else 'ATLAS_proportion_'
    features[name] = [f for f in names if f.startswith(prefix)]
    assert len(set(features[name])) == len(features[name]) == (52 if name=='gep' else 11)
    cuts[name] = resolve_source(name, features[name])
    audit = """
import sys, pandas as pd
from survival.survival_model import load_tcat_analysis
d, _, _, weights, _ = load_tcat_analysis(*sys.argv[1:4], 1000, ['Treatment','Disease'])
keys = ['Dataset','sample_id']
assert set(map(tuple,d[keys].values)) == set(map(tuple,pd.read_csv(sys.argv[4])[keys].values)), 'Patient identities differ'
assert weights.shape==(163,16)
"""
    subprocess.run([sys.executable,'-c',audit,csv,npz,outcomes,oe],cwd=repo,check=True)
    print(name, '| field features:', len(features[name]), '| individual crossover table:', cuts[name])

In [ ]:
def run_joint(name):
    csv,npz = inputs[name]
    paths = [csv,npz,outcomes,oe,op,cuts[name],o_cut]
    kind = 'gep' if name=='gep' else 'cell_type'
    flags = ['--feature-kind',kind,'--midpoint-centered','--transition-scale-sd',str(width),
             '--minimum-cells','1000','--draws',str(draws),'--tune',str(tune),
             '--chains',str(chains),'--cores',str(cores),'--target-accept',str(target_accept),
             '--random-state',str(seed),'--adjustment','Treatment=axicel','--adjustment','Disease=lymphoma']
    config = dict(revision=revision, flags=flags, input_sha256=[digest(path) for path in paths])
    tag = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:12]
    dest = OUTPUT/name/tag
    if not dest.exists():
        dest.parent.mkdir(parents=True,exist_ok=True)
        stage = Path(tempfile.mkdtemp(prefix=tag+'-running-',dir=dest.parent))
        command = [sys.executable,'-u','-m','survival.fit_tcat_joint_survival_regimes',*paths,stage,*flags]
        print('Fitting',name,'|',dest,flush=True)
        with subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1) as proc:
            for line in proc.stdout:
                print(line,end='',flush=True)
            if proc.wait():
                raise subprocess.CalledProcessError(proc.returncode, command)
        (stage/'notebook_config.json').write_text(json.dumps(config,indent=2))
        stage.rename(dest)
    assert json.loads((dest/'notebook_config.json').read_text())==config
    assert json.loads((dest/'joint_manifest.json').read_text())['hazard_activation_offset']==0.5
    frame = pd.read_csv(dest/'joint_feature_results.csv')
    assert set(frame.feature)==set(features[name])|{'O_lambda_inf'} and frame.feature.is_unique
    assert frame.patients.eq(163).all() and frame.datasets.eq(8).all()
    assert (dest/'posterior.nc').stat().st_size
    print('Saved:',dest)
    display(frame[['feature','HR_median','HR_q025','HR_q975','crossover_median_fixed',
                   'divergences','max_rhat','min_ess_bulk']])
    return dest

if 'gep' in RUN:
    gep_output = run_joint('gep')

In [ ]:
if 'atlas' in RUN:
    atlas_output = run_joint('atlas')